# 06 - Assemble the whole library

Reads the four standardised candidate tables (01, 02, 03+04) plus the barcode
table (05), attaches one barcode per candidate, and writes the full ordered
sequence for every member of the library.

```
BG5 + promoter + RE1 + RE2 + BARCODE + BG3
```

The promoter enters at its own natural length: 69 nt for the designed set,
50-66 nt for native, 75 nt for both phage sources. Only the flanks are shared,
and they are added here and nowhere else.

**Input** `outputs/01_final_design.csv`, `02_native_promoters.csv`,
`03_phage_promoters.csv`, `04_phage_promoters.csv`, `05_barcodes.csv`

**Output** `outputs/06_whole_sequence.csv`

In [ ]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


## Construct definition

The only place the flanks are defined. Change an enzyme here, then re-run 06 and 07.

In [ ]:
# === Construct constants ===
BG5 = "CACGAGGCCCTTTCGTCTTCACACAGCAGCAGTCAGGTAGGGAAGAGACC"
RE1 = "GTCGAC"      # SalI
RE2 = "TCTAGA"      # XbaI
BG3 = "GGTCTCAAGGCTGCTAACAAAGCCCGAAAGGAAGCTGAGTTGGCTGCTGC"
BARCODE_LEN = 15

# Set True to also assemble candidates whose source table marked qc_pass False
# (for example native rows carrying '*'). They stay flagged either way.
INCLUDE_QC_FAIL = False

for _name, _seq in [("BG5", BG5), ("RE1", RE1), ("RE2", RE2), ("BG3", BG3)]:
    if not set(_seq) <= set("ACGT"):
        raise ValueError(f"{_name} contains non-ACGT characters")

CONSTANT_LEN = len(BG5) + len(RE1) + len(RE2) + BARCODE_LEN + len(BG3)
print(f"BG5 {len(BG5)} | RE1 {len(RE1)} | RE2 {len(RE2)} | barcode {BARCODE_LEN} | BG3 {len(BG3)}")
print(f"constant length = {CONSTANT_LEN} nt (promoter adds on top of this)")


## Load the standardised candidate tables

In [ ]:
# === Load 01-05 ===
import pandas as pd

SOURCES = {
    "01_final_design.csv":     "designed",
    "02_native_promoters.csv": "native",
    "03_phage_promoters.csv":  "phage scan",
    "04_phage_promoters.csv":  "phage curated",
}

# Provenance columns worth carrying into the assembled table. Everything else
# stays in the per-source CSV and can be joined back on candidate_id.
CARRY = [
    "organism", "native_locus_tag", "native_kegg_pool", "native_kegg_category",
    "phage_species", "phage_host", "phage_sigma_factor", "phage_genome_accession",
    "phage_promoter_window", "method",
    "design_run", "register_clean", "design_core_score", "m35_shift", "m10_shift",
]

frames = []
for fname, label in SOURCES.items():
    path = require(RELEASE_OUT / fname, f"{label} candidates")
    df = pd.read_csv(path)
    missing = [c for c in STD_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{fname} is missing standard columns: {missing}")
    frames.append(df[STD_COLS + [c for c in CARRY if c in df.columns]])
    print(f"{label:16s} {len(df):>6d} rows   qc_pass={int(df['qc_pass'].sum()):>6d}   "
          f"len {int(df['promoter_length'].min())}-{int(df['promoter_length'].max())}")

candidates = pd.concat(frames, ignore_index=True)
print(f"\ntotal candidates: {len(candidates)}")

if candidates["candidate_id"].duplicated().any():
    dup = candidates.loc[candidates["candidate_id"].duplicated(), "candidate_id"].head().tolist()
    raise ValueError(f"candidate_id collision across sources: {dup}")

barcodes = pd.read_csv(require(RELEASE_OUT / "05_barcodes.csv", "barcodes"))
observed = sorted(barcodes["barcode"].str.len().unique())
if observed != [BARCODE_LEN]:
    raise ValueError(f"05_barcodes.csv holds {observed} nt barcodes, construct expects {BARCODE_LEN}")
print(f"barcodes available: {len(barcodes)} x {BARCODE_LEN} nt")


## Attach barcodes

One barcode per candidate, assigned in `candidate_id` order so the mapping is
reproducible across re-runs.

In [ ]:
# === Barcode assignment ===
lib = candidates if INCLUDE_QC_FAIL else candidates[candidates["qc_pass"]].copy()
lib = lib.sort_values("candidate_id").reset_index(drop=True)
print(f"assembling {len(lib)} candidates (INCLUDE_QC_FAIL={INCLUDE_QC_FAIL})")

if len(lib) > len(barcodes):
    raise ValueError(
        f"not enough barcodes: {len(lib)} candidates vs {len(barcodes)} barcodes. "
        f"Raise N_BARCODES in 05 and re-run it."
    )

lib["barcode_id"] = barcodes["candidate_id"].values[: len(lib)]
lib["barcode"] = barcodes["barcode"].values[: len(lib)]
print(f"barcodes used: {len(lib)} / {len(barcodes)}  ({len(barcodes) - len(lib)} spare)")


## Assemble

Every segment offset is recorded, so 07 can attribute each restriction site to
the part of the construct it came from instead of just counting hits.

In [ ]:
# === Build full_sequence and segment offsets ===
promoter = lib["promoter_sequence"].astype(str)

lib["full_sequence"] = BG5 + promoter + RE1 + RE2 + lib["barcode"].astype(str) + BG3
lib["full_length"] = lib["full_sequence"].str.len()

# 0-based, end-exclusive. Only the promoter varies in length, so everything
# downstream of it shifts per row.
lib["bg5_start"] = 0
lib["bg5_end"] = len(BG5)
lib["promoter_start"] = len(BG5)
lib["promoter_end"] = len(BG5) + lib["promoter_length"]
lib["re1_start"] = lib["promoter_end"]
lib["re2_start"] = lib["re1_start"] + len(RE1)
lib["barcode_start"] = lib["re2_start"] + len(RE2)
lib["bg3_start"] = lib["barcode_start"] + BARCODE_LEN
lib["bg3_end"] = lib["bg3_start"] + len(BG3)

lib["bg5_seq"] = BG5
lib["re1_seq"] = RE1
lib["re2_seq"] = RE2
lib["bg3_seq"] = BG3

# Independent reconstruction: slicing the assembled sequence by the recorded
# offsets must give back exactly the parts that went in.
ok = (
    (lib["full_length"] == lib["bg3_end"])
    & lib.apply(lambda r: r["full_sequence"][r["promoter_start"]:r["promoter_end"]] == r["promoter_sequence"], axis=1)
    & lib.apply(lambda r: r["full_sequence"][r["barcode_start"]:r["barcode_start"] + BARCODE_LEN] == r["barcode"], axis=1)
    & lib["full_sequence"].str.startswith(BG5)
    & lib["full_sequence"].str.endswith(BG3)
)
if not ok.all():
    raise ValueError(f"{int((~ok).sum())} rows failed the offset round-trip check")
print("offset round-trip check passed for all", len(lib), "rows")

print("\nfull_length by source:")
print(lib.groupby("source")["full_length"].agg(["count", "min", "max"]).to_string())


## Write `06_whole_sequence.csv`

In [ ]:
# === Output ===
ORDER = (
    STD_COLS
    + ["barcode_id", "barcode", "full_sequence", "full_length"]
    + ["bg5_start", "bg5_end", "promoter_start", "promoter_end",
       "re1_start", "re2_start", "barcode_start", "bg3_start", "bg3_end"]
    + ["bg5_seq", "re1_seq", "re2_seq", "bg3_seq"]
)
ORDER += [c for c in lib.columns if c not in ORDER]
out = lib[ORDER]

dest = RELEASE_OUT / "06_whole_sequence.csv"
out.to_csv(dest, index=False)
print(f"wrote {dest}  {out.shape}")

print("\nper-source counts:")
print(out["source"].value_counts().to_string())
print("\npromoter length by source:")
print(out.groupby("source")["promoter_length"].agg(["min", "max"]).to_string())
print("\nfirst rows:")
print(out[["candidate_id", "source", "promoter_length", "full_length", "barcode_id"]].head(5).to_string(index=False))
print("\nNext: run 07_re_scan.ipynb on this file.")
